# StutterSense: Annotator-Aware Stuttering Event Detection

Multi-label detection of five stuttering events from frozen WavLM Base+ embeddings. The experiment compares majority labels, annotator proportions, Brier regularization, and supervised ambiguity prediction. Results are exploratory and not clinical diagnoses.

## 1. Environment

Colab installs missing dependencies; local Jupyter validates `requirements.txt`.

In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules
COLAB_PACKAGES = [
    "datasets>=3.2,<5", "transformers>=4.48,<6", "accelerate>=1.2,<2",
    "huggingface-hub>=0.27,<2", "soundfile>=0.12,<1", "librosa>=0.10.2,<1",
    "tqdm>=4.66", "ipywidgets>=8,<9",
]

def installed_version(distribution):
    try:
        return version(distribution)
    except PackageNotFoundError:
        return None

if IN_COLAB:
    preloaded_core = {
        distribution: installed_version(distribution)
        for module, distribution in {
            "numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
            "sklearn": "scikit-learn", "matplotlib": "matplotlib", "seaborn": "seaborn",
        }.items()
        if module in sys.modules
    }
    if importlib.util.find_spec("torch") is None:
        COLAB_PACKAGES.insert(0, "torch>=2.2,<3")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *COLAB_PACKAGES])
    changed_core = {
        name: (before, installed_version(name))
        for name, before in preloaded_core.items()
        if before != installed_version(name)
    }
    if changed_core:
        raise RuntimeError(
            f"Colab changed preloaded scientific packages: {changed_core}. "
            "Restart the runtime once, then run all cells again."
        )
else:
    required = {
        "datasets": "datasets", "transformers": "transformers", "accelerate": "accelerate",
        "huggingface_hub": "huggingface-hub", "soundfile": "soundfile", "librosa": "librosa",
        "scipy": "scipy", "sklearn": "scikit-learn", "pandas": "pandas", "numpy": "numpy",
        "matplotlib": "matplotlib", "seaborn": "seaborn", "tqdm": "tqdm",
        "ipywidgets": "ipywidgets", "torch": "torch",
    }
    missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
    if missing:
        raise ModuleNotFoundError(
            "Missing dependencies: " + ", ".join(missing)
            + ". Run: python -m pip install -r requirements.txt"
        )
print("Environment ready:", "Google Colab" if IN_COLAB else sys.executable)

In [ ]:
import hashlib
import io
import json
import math
import os
import platform
import random
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path

import datasets
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn
import soundfile as sf
import torch
import transformers
from datasets import Audio, load_dataset
from IPython.display import Audio as IPythonAudio, FileLink, Markdown, clear_output, display
from huggingface_hub import HfApi
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score, f1_score, precision_recall_curve,
    precision_recall_fscore_support, roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, WavLMModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | device={DEVICE}")
print(f"transformers {transformers.__version__} | datasets {datasets.__version__}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

The default run samples 2,000 clips from the complete stream. Smoke modes are for debugging only.

In [ ]:
DATASET_ID = "psusac/sep28k"
DATASET_CONFIG = "default"
DATASET_REVISION = None
MODEL_NAME = "microsoft/wavlm-base-plus"
MODEL_REVISION = None
MAX_SAMPLES = 2000
SAMPLE_RATE = 16000
RANDOM_SEED = 42
EXPERIMENT_SEEDS = [42]
BATCH_SIZE = 8
EPOCHS = 12
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 3
DROPOUT = 0.20
OUTPUT_DIR = os.environ.get("STUTTERSENSE_OUTPUT_DIR", "/content/stuttersense_outputs" if IN_COLAB else "outputs")

FAST_SMOKE_TEST = False
CPU_SMOKE_TEST = False
SMOKE_STREAM_SCAN_LIMIT = 512
BRIER_LAMBDA = 0.25
ENABLE_AMBIGUITY_HEAD = True
AMBIGUITY_LOSS_WEIGHT = 0.25
RUN_QUALITY_FILTER_ABLATION = False
QUALITY_FILTER_MIN_COUNT = 1
BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_CONFIDENCE = 0.95
GROUP_COLUMN_OVERRIDE = None
SHOW_OUTPUT_ZIP_LINK = True

ENABLE_EXTERNAL_FLUENCYBANK = False
FLUENCYBANK_MANIFEST_PATH = None
MAX_UPLOAD_MB = 100
MAX_UPLOAD_TOTAL_SECONDS = 300
UPLOAD_CHUNK_SECONDS = 10
UPLOAD_CHUNK_OVERLAP_SECONDS = 1

EVENTS = ["prolongation", "block", "sound_rep", "word_rep", "interjection"]
DIAGNOSTIC_COLUMNS = [
    "unsure", "poor_audio_quality", "no_speech", "difficult_to_understand",
    "no_stuttered_words", "natural_pause", "music",
]
VERIFIED_GROUP_COLUMNS = ["speaker_id", "speaker", "episode_id", "episode"]
SAMPLE_IDENTIFIER_COLUMNS = ["clip_id", "sample_id", "file_id", "utterance_id"]

RUN_MAX_SAMPLES = MAX_SAMPLES
RUN_BATCH_SIZE = BATCH_SIZE
RUN_EPOCHS = EPOCHS
RUN_STREAM_SCAN_LIMIT = None
RUN_BOOTSTRAP_REPLICATES = BOOTSTRAP_REPLICATES
if FAST_SMOKE_TEST:
    RUN_MAX_SAMPLES = 64 if MAX_SAMPLES is None else min(MAX_SAMPLES, 64)
    RUN_BATCH_SIZE = min(BATCH_SIZE, 4)
    RUN_EPOCHS = min(EPOCHS, 2)
    RUN_STREAM_SCAN_LIMIT = SMOKE_STREAM_SCAN_LIMIT
    RUN_BOOTSTRAP_REPLICATES = min(BOOTSTRAP_REPLICATES, 100)
if CPU_SMOKE_TEST:
    DEVICE = torch.device("cpu")
    RUN_MAX_SAMPLES = 32 if RUN_MAX_SAMPLES is None else min(RUN_MAX_SAMPLES, 32)
    RUN_BATCH_SIZE = min(RUN_BATCH_SIZE, 2)
    RUN_EPOCHS = 1
    RUN_STREAM_SCAN_LIMIT = min(SMOKE_STREAM_SCAN_LIMIT, 256)
    RUN_BOOTSTRAP_REPLICATES = min(BOOTSTRAP_REPLICATES, 50)

OUTPUT_PATH = Path(OUTPUT_DIR)
FIGURE_PATH = OUTPUT_PATH / "figures"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
print({"device": str(DEVICE), "samples": RUN_MAX_SAMPLES, "seeds": EXPERIMENT_SEEDS,
       "stream_scan_limit": RUN_STREAM_SCAN_LIMIT, "output": str(OUTPUT_PATH.resolve())})

## 3. Dataset and sampling

Dataset and model revisions are pinned. Algorithm R reservoir sampling gives every streamed row equal inclusion probability.

In [ ]:
hub = HfApi()
dataset_hub_info = hub.dataset_info(DATASET_ID, revision=DATASET_REVISION)
model_hub_info = hub.model_info(MODEL_NAME, revision=MODEL_REVISION)
RESOLVED_DATASET_REVISION = dataset_hub_info.sha
RESOLVED_MODEL_REVISION = model_hub_info.sha
def card_license(card_data):
    if card_data is None:
        return None
    return card_data.get("license") if isinstance(card_data, dict) else getattr(card_data, "license", None)

DATASET_CARD_LICENSE = card_license(dataset_hub_info.card_data)
MODEL_CARD_LICENSE = card_license(model_hub_info.card_data)
print("Dataset revision:", RESOLVED_DATASET_REVISION)
print("Model revision:", RESOLVED_MODEL_REVISION)

raw_stream = load_dataset(
    DATASET_ID, DATASET_CONFIG, split="train", streaming=True,
    revision=RESOLVED_DATASET_REVISION,
)
if raw_stream.features is None:
    raise RuntimeError("The hosted dataset did not expose a schema.")
available_columns = list(raw_stream.features)
missing_columns = [name for name in ["audio", *EVENTS] if name not in available_columns]
if missing_columns:
    raise RuntimeError(f"Required columns are missing: {missing_columns}")

verified_identifier_columns = [
    name for name in [*SAMPLE_IDENTIFIER_COLUMNS, *VERIFIED_GROUP_COLUMNS]
    if name in available_columns
]
print("Hosted schema:", raw_stream.features)
print("Verified identifier/group columns:", verified_identifier_columns or "none")
raw_stream = raw_stream.cast_column("audio", Audio(decode=False))

In [ ]:
def reservoir_sample(iterable, sample_size, seed, scan_limit=None):
    """Uniformly sample without replacement while scanning the complete iterable."""
    rng = np.random.default_rng(seed)
    reservoir = []
    scanned = 0
    for position, item in enumerate(iterable):
        if scan_limit is not None and position >= scan_limit:
            break
        if sample_size is None:
            reservoir.append((position, item))
        elif len(reservoir) < sample_size:
            reservoir.append((position, item))
        else:
            replacement = int(rng.integers(0, position + 1))
            if replacement < sample_size:
                reservoir[replacement] = (position, item)
        scanned += 1
    reservoir.sort(key=lambda pair: pair[0])
    return reservoir, scanned

def decode_audio_value(value):
    """Return mono float32 samples and the source sample rate."""
    if hasattr(value, "get_all_samples"):
        decoded = value.get_all_samples()
        array = decoded.data.detach().cpu().numpy() if torch.is_tensor(decoded.data) else np.asarray(decoded.data)
        source_rate = int(decoded.sample_rate)
    elif isinstance(value, dict) and value.get("array") is not None:
        array = np.asarray(value["array"])
        source_rate = int(value["sampling_rate"])
    else:
        payload = None
        if isinstance(value, dict):
            payload = io.BytesIO(value["bytes"]) if value.get("bytes") is not None else value.get("path")
        elif isinstance(value, (bytes, bytearray)):
            payload = io.BytesIO(value)
        elif isinstance(value, (str, os.PathLike)):
            payload = str(value)
        if payload is None:
            raise TypeError(f"Unsupported audio value: {type(value)!r}")
        try:
            array, source_rate = sf.read(payload, dtype="float32", always_2d=False)
        except Exception:
            if hasattr(payload, "seek"):
                payload.seek(0)
            array, source_rate = librosa.load(payload, sr=None, mono=False)

    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        array = array.mean(axis=0 if array.shape[0] <= array.shape[1] else 1)
    if array.ndim != 1 or array.size == 0 or not np.isfinite(array).all():
        raise ValueError(f"Invalid waveform shape or values: {array.shape}")
    if int(source_rate) <= 0:
        raise ValueError(f"Invalid sample rate: {source_rate}")
    if int(source_rate) != SAMPLE_RATE:
        array = librosa.resample(array, orig_sr=int(source_rate), target_sr=SAMPLE_RATE)
    return np.ascontiguousarray(array, dtype=np.float32), int(source_rate)

sampled_raw, stream_examples_scanned = reservoir_sample(
    tqdm(raw_stream, desc="Scanning hosted stream"), RUN_MAX_SAMPLES,
    RANDOM_SEED, RUN_STREAM_SCAN_LIMIT,
)
if RUN_MAX_SAMPLES is not None and stream_examples_scanned < RUN_MAX_SAMPLES:
    warnings.warn(f"Requested {RUN_MAX_SAMPLES} samples but the scanned stream contained {stream_examples_scanned}.")

records, waveforms, decode_failures = [], [], []
for stream_position, example in tqdm(sampled_raw, desc="Decoding sampled clips"):
    try:
        counts = np.asarray([example[name] for name in EVENTS], dtype=np.float32)
        if not np.isfinite(counts).all() or np.any(counts < 0) or np.any(counts > 3):
            raise ValueError(f"Annotation counts outside [0, 3]: {counts.tolist()}")
        if not np.allclose(counts, np.round(counts)):
            raise ValueError(f"Non-integer annotation counts: {counts.tolist()}")
        waveform, original_rate = decode_audio_value(example["audio"])
        audio_sha256 = hashlib.sha256(waveform.tobytes()).hexdigest()
        row = {name: int(example[name]) for name in EVENTS}
        for name in DIAGNOSTIC_COLUMNS:
            if name in example and example[name] is not None:
                row[name] = int(example[name])
        for name in verified_identifier_columns:
            if example.get(name) is not None:
                row[name] = str(example[name])
        row.update({
            "stream_position": int(stream_position),
            "original_sample_rate": original_rate, "num_samples": int(waveform.size),
            "duration_seconds": float(waveform.size / SAMPLE_RATE), "audio_sha256": audio_sha256,
        })
        if "sample_id" not in row:
            row["sample_id"] = audio_sha256
        records.append(row)
        waveforms.append(waveform)
    except Exception as exc:
        decode_failures.append({"stream_position": int(stream_position), "error": repr(exc)})

if not records:
    raise RuntimeError("No sampled clips decoded successfully.")
metadata = pd.DataFrame(records)
counts_matrix = metadata[EVENTS].to_numpy(dtype=np.float32)
hard_targets = (counts_matrix >= 2).astype(np.float32)
soft_targets = counts_matrix / 3.0
ambiguity_targets = np.isin(counts_matrix, [1, 2]).astype(np.float32)
durations = metadata["duration_seconds"].to_numpy()
unusual_mask = (durations < 0.25) | (durations > 30.0)

sampling_metadata = {
    "algorithm": "Algorithm R reservoir sampling without replacement",
    "seed": RANDOM_SEED, "stream_examples_scanned": stream_examples_scanned,
    "scan_limit": RUN_STREAM_SCAN_LIMIT, "requested_samples": RUN_MAX_SAMPLES,
    "decoded_samples": len(metadata), "decode_failures": len(decode_failures),
    "full_stream": RUN_STREAM_SCAN_LIMIT is None,
}
print(sampling_metadata)
print(metadata["duration_seconds"].describe())
print("Exact duplicate fingerprints:", int(metadata["audio_sha256"].duplicated().sum()))
print("Unusual-duration clips:", int(unusual_mask.sum()))
if decode_failures:
    display(pd.DataFrame(decode_failures).head(20))
display(metadata.head())

In [ ]:
print("A few streamed examples (playback is optional):")
for i in range(min(3, len(waveforms))):
    display(Markdown(f"**Clip {i}** — {metadata.loc[i, EVENTS].to_dict()}"))
    display(IPythonAudio(waveforms[i], rate=SAMPLE_RATE))

## 4. Data overview

Counts 1–2 indicate aggregate ambiguity; co-occurrence uses majority labels.

In [ ]:
def save_current_figure(filename):
    path = FIGURE_PATH / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved", path)

prevalence = pd.Series(hard_targets.mean(axis=0), index=EVENTS)
ax = prevalence.sort_values().plot(kind="barh", color="#3B82F6", figsize=(8, 4.5))
ax.set(xlabel="Majority-vote prevalence", ylabel="Event", title="A. Dataset label frequency")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
save_current_figure("A_dataset_label_frequency.png")
plt.show()

count_long = metadata[EVENTS].melt(var_name="event", value_name="annotation_count")
count_table = pd.crosstab(count_long["event"], count_long["annotation_count"]).reindex(columns=[0, 1, 2, 3], fill_value=0)
count_table.plot(kind="bar", figsize=(10, 5), color=["#CBD5E1", "#FBBF24", "#F97316", "#2563EB"])
plt.title("Annotation count distribution")
plt.xlabel("Event")
plt.ylabel("Number of clips")
plt.legend(title="Count out of 3")
plt.xticks(rotation=25, ha="right")
save_current_figure("annotation_count_distribution.png")
plt.show()

cooccurrence = pd.DataFrame(hard_targets, columns=EVENTS).corr().fillna(0)
plt.figure(figsize=(7, 6))
sns.heatmap(cooccurrence, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, square=True)
plt.title("Majority-label event co-occurrence (phi correlation)")
save_current_figure("event_cooccurrence.png")
plt.show()

plt.figure(figsize=(8, 4.5))
sns.histplot(durations, bins=30, color="#2563EB")
plt.xlabel("Duration (seconds)")
plt.title("Audio duration distribution (no truncation)")
save_current_figure("audio_duration_distribution.png")
plt.show()

ambiguity_by_event = pd.Series(((counts_matrix == 1) | (counts_matrix == 2)).mean(axis=0), index=EVENTS)
ax = ambiguity_by_event.sort_values().plot(kind="barh", color="#F59E0B", figsize=(8, 4.5))
ax.set(xlabel="Fraction with count 1 or 2", ylabel="Event", title="B. Annotator disagreement proxy")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
save_current_figure("B_annotator_disagreement.png")
plt.show()

display(pd.DataFrame({"majority_prevalence": prevalence, "ambiguity_rate": ambiguity_by_event}))

## 5. Splits and quality flags

Exact duplicates stay together. Verified speaker or episode IDs are used only when present; no identities are inferred.

In [ ]:
def select_verified_group_column(frame, preferred_columns, override=None):
    """Select only an explicitly named speaker/episode column with usable values."""
    if override is not None:
        if override not in frame.columns:
            raise ValueError(f"GROUP_COLUMN_OVERRIDE={override!r} is not a hosted column.")
        if override not in preferred_columns:
            raise ValueError("The override must be explicitly listed in VERIFIED_GROUP_COLUMNS.")
        candidates = [override]
    else:
        candidates = [name for name in preferred_columns if name in frame.columns]
    for name in candidates:
        values = frame[name]
        if values.notna().all() and (values.astype(str).str.len() > 0).all() and values.nunique() >= 3:
            return name
    return None

def build_dependence_groups(audio_hashes, identity_values=None):
    """Return connected groups induced by exact duplicates and verified identities."""
    audio_hashes = np.asarray(audio_hashes, dtype=object)
    n = len(audio_hashes)
    parent = np.arange(n)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for values in [audio_hashes, None if identity_values is None else np.asarray(identity_values, dtype=object)]:
        if values is None:
            continue
        first = {}
        for i, value in enumerate(values):
            key = str(value)
            if key in first:
                union(i, first[key])
            else:
                first[key] = i
    roots = [find(i) for i in range(n)]
    labels = {root: index for index, root in enumerate(dict.fromkeys(roots))}
    return np.asarray([labels[root] for root in roots])

verified_group_column = select_verified_group_column(
    metadata, VERIFIED_GROUP_COLUMNS, GROUP_COLUMN_OVERRIDE,
)
identity_values = metadata[verified_group_column].to_numpy() if verified_group_column else None
dependence_groups = build_dependence_groups(metadata["audio_sha256"].to_numpy(), identity_values)
metadata["dependence_group"] = dependence_groups
split_scope = (
    f"verified {verified_group_column}-group split plus duplicate protection"
    if verified_group_column else "exact-duplicate-aware clip split; speaker leakage possible"
)
print("Split scope:", split_scope)

all_indices = np.arange(len(metadata))
if np.unique(dependence_groups).size < 3:
    raise RuntimeError("At least three dependence groups are required.")
outer = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=RANDOM_SEED)
train_idx, temp_idx = next(outer.split(all_indices, groups=dependence_groups))
inner = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=RANDOM_SEED + 1)
val_rel, test_rel = next(inner.split(temp_idx, groups=dependence_groups[temp_idx]))
val_idx, test_idx = temp_idx[val_rel], temp_idx[test_rel]
split_indices = {
    "train": np.sort(train_idx).tolist(), "validation": np.sort(val_idx).tolist(),
    "test": np.sort(test_idx).tolist(),
}
for left, right in ((train_idx, val_idx), (train_idx, test_idx), (val_idx, test_idx)):
    assert set(dependence_groups[left]).isdisjoint(set(dependence_groups[right]))

quality_columns_present = [name for name in DIAGNOSTIC_COLUMNS if name in metadata.columns and name != "no_stuttered_words"]
quality_flag_matrix = metadata[quality_columns_present].fillna(0).to_numpy() >= QUALITY_FILTER_MIN_COUNT
quality_any_flag = quality_flag_matrix.any(axis=1) if quality_columns_present else np.zeros(len(metadata), dtype=bool)
quality_filtered_train_idx = train_idx[~quality_any_flag[train_idx]]

coverage_rows = []
for split_name, indices in split_indices.items():
    for event_i, event in enumerate(EVENTS):
        coverage_rows.append({
            "split": split_name, "n_clips": len(indices), "event": event,
            "positives": int(hard_targets[indices, event_i].sum()),
            "prevalence": float(hard_targets[indices, event_i].mean()),
        })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
print({name: len(indices) for name, indices in split_indices.items()})
print(f"Quality-filtered training clips: {len(quality_filtered_train_idx)} / {len(train_idx)}")

## 6. WavLM embeddings

Frozen WavLM features use attention-masked mean pooling and a revision-keyed cache.

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME, revision=RESOLVED_MODEL_REVISION)
encoder = WavLMModel.from_pretrained(MODEL_NAME, revision=RESOLVED_MODEL_REVISION)
encoder.requires_grad_(False)
encoder.eval().to(DEVICE)

cache_key_payload = {
    "model": MODEL_NAME,
    "model_revision": RESOLVED_MODEL_REVISION,
    "dataset_revision": RESOLVED_DATASET_REVISION,
    "sample_rate": SAMPLE_RATE,
    "fingerprints": metadata["audio_sha256"].tolist(),
}
cache_key = hashlib.sha256(json.dumps(cache_key_payload, sort_keys=True).encode()).hexdigest()[:16]
embedding_cache = OUTPUT_PATH / f"wavlm_embeddings_{cache_key}.npz"

def masked_mean_pool(hidden_states, input_attention_mask, model):
    if hasattr(model, "_get_feature_vector_attention_mask"):
        frame_mask = model._get_feature_vector_attention_mask(hidden_states.shape[1], input_attention_mask)
    else:
        input_lengths = input_attention_mask.sum(dim=-1)
        if hasattr(model, "_get_feat_extract_output_lengths"):
            output_lengths = model._get_feat_extract_output_lengths(input_lengths)
        else:
            output_lengths = torch.ceil(
                input_lengths.float() * hidden_states.shape[1] / input_attention_mask.shape[1]
            ).long()
        positions = torch.arange(hidden_states.shape[1], device=hidden_states.device)[None, :]
        frame_mask = positions < output_lengths[:, None]
    weights = frame_mask.unsqueeze(-1).to(hidden_states.dtype)
    return (hidden_states * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)

if embedding_cache.exists():
    cached = np.load(embedding_cache)
    embeddings = cached["embeddings"]
    if embeddings.shape[0] != len(waveforms):
        raise RuntimeError("Embedding cache length mismatch.")
    print("Loaded cached embeddings:", embedding_cache, embeddings.shape)
else:
    embedding_chunks = []
    autocast_enabled = DEVICE.type == "cuda"
    with torch.inference_mode():
        for start in tqdm(range(0, len(waveforms), RUN_BATCH_SIZE), desc="Extracting WavLM embeddings"):
            batch_audio = waveforms[start:start + RUN_BATCH_SIZE]
            inputs = feature_extractor(
                batch_audio,
                sampling_rate=SAMPLE_RATE,
                padding=True,
                truncation=False,
                return_attention_mask=True,
                return_tensors="pt",
            )
            input_values = inputs["input_values"].to(DEVICE)
            attention_mask = inputs.get("attention_mask", torch.ones_like(input_values, dtype=torch.long)).to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=autocast_enabled):
                hidden = encoder(input_values=input_values, attention_mask=attention_mask).last_hidden_state
                pooled = masked_mean_pool(hidden, attention_mask, encoder)
            embedding_chunks.append(pooled.float().cpu().numpy())
    embeddings = np.concatenate(embedding_chunks).astype(np.float32)
    np.savez_compressed(embedding_cache, embeddings=embeddings)
    print("Saved shared embedding cache:", embedding_cache, embeddings.shape)

assert embeddings.shape == (len(waveforms), encoder.config.hidden_size)
encoder.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Models and losses

Comparisons include hard BCE, soft BCE, a Brier-regularization ablation, and an optional ambiguity head trained on counts 1–2.

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, x, hard_y, soft_y, ambiguity_y, indices):
        self.x = torch.from_numpy(x[indices]).float()
        self.hard_y = torch.from_numpy(hard_y[indices]).float()
        self.soft_y = torch.from_numpy(soft_y[indices]).float()
        self.ambiguity_y = torch.from_numpy(ambiguity_y[indices]).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], self.hard_y[index], self.soft_y[index], self.ambiguity_y[index]

class StutterMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=5, dropout=DROPOUT):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.network(x)

class AmbiguityMultiTaskMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=5, dropout=DROPOUT):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.event_head = nn.Linear(hidden_dim, output_dim)
        self.ambiguity_head = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        hidden = self.shared(x)
        return self.event_head(hidden), self.ambiguity_head(hidden)

def make_loader(indices, seed, shuffle=False):
    dataset = EmbeddingDataset(embeddings, hard_targets, soft_targets, ambiguity_targets, np.asarray(indices))
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=RUN_BATCH_SIZE, shuffle=shuffle,
        generator=generator if shuffle else None, num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )

def soft_brier_regularized_loss(logits, soft_y, brier_lambda):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, soft_y)
    brier = torch.mean((torch.sigmoid(logits) - soft_y) ** 2)
    return bce + brier_lambda * brier

def ambiguity_multitask_loss(event_logits, ambiguity_logits, soft_y, ambiguity_y, ambiguity_weight):
    event_loss = nn.functional.binary_cross_entropy_with_logits(event_logits, soft_y)
    ambiguity_loss = nn.functional.binary_cross_entropy_with_logits(ambiguity_logits, ambiguity_y)
    return event_loss + ambiguity_weight * ambiguity_loss

def training_objective(outputs, hard_y, soft_y, ambiguity_y, variant):
    if variant == "ambiguity_multitask":
        return ambiguity_multitask_loss(*outputs, soft_y, ambiguity_y, AMBIGUITY_LOSS_WEIGHT)
    logits = outputs
    if variant == "hard":
        return nn.functional.binary_cross_entropy_with_logits(logits, hard_y)
    if variant in {"soft", "soft_quality_filtered"}:
        return nn.functional.binary_cross_entropy_with_logits(logits, soft_y)
    if variant == "brier_ablation":
        return soft_brier_regularized_loss(logits, soft_y, BRIER_LAMBDA)
    raise ValueError(f"Unknown variant: {variant}")

print(StutterMLP(embeddings.shape[1]))

## 8. Training

Seeds share one split and embedding cache. Early stopping uses validation loss.

In [ ]:
def train_variant(variant, display_name, run_seed, train_indices):
    set_seed(run_seed)
    model_class = AmbiguityMultiTaskMLP if variant == "ambiguity_multitask" else StutterMLP
    model = model_class(embeddings.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    train_loader = make_loader(train_indices, run_seed, shuffle=True)
    val_loader = make_loader(val_idx, run_seed, shuffle=False)
    checkpoint_dir = OUTPUT_PATH / "checkpoints" / f"seed_{run_seed}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / f"{variant}_best.pt"
    history = {"train_loss": [], "validation_loss": []}
    best_val, stale_epochs = float("inf"), 0

    for epoch in range(RUN_EPOCHS):
        model.train()
        running, seen = 0.0, 0
        progress = tqdm(train_loader, desc=f"seed {run_seed} | {display_name} | {epoch + 1}/{RUN_EPOCHS}", leave=False)
        for x, hard_y, soft_y, ambiguity_y in progress:
            x, hard_y = x.to(DEVICE), hard_y.to(DEVICE)
            soft_y, ambiguity_y = soft_y.to(DEVICE), ambiguity_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = training_objective(model(x), hard_y, soft_y, ambiguity_y, variant)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(x)
            seen += len(x)
        train_loss = running / seen

        model.eval()
        val_running, val_seen = 0.0, 0
        with torch.inference_mode():
            for x, hard_y, soft_y, ambiguity_y in val_loader:
                x, hard_y = x.to(DEVICE), hard_y.to(DEVICE)
                soft_y, ambiguity_y = soft_y.to(DEVICE), ambiguity_y.to(DEVICE)
                loss = training_objective(model(x), hard_y, soft_y, ambiguity_y, variant)
                val_running += loss.item() * len(x)
                val_seen += len(x)
        val_loss = val_running / val_seen
        history["train_loss"].append(train_loss)
        history["validation_loss"].append(val_loss)
        print(f"seed {run_seed} | {display_name} | epoch {epoch + 1:02d} | train {train_loss:.5f} | val {val_loss:.5f}")

        if val_loss < best_val - 1e-6:
            best_val, stale_epochs = val_loss, 0
            torch.save({
                "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "variant": variant, "input_dim": embeddings.shape[1], "events": EVENTS,
                "best_validation_loss": best_val, "seed": run_seed,
                "dataset_revision": RESOLVED_DATASET_REVISION, "model_revision": RESOLVED_MODEL_REVISION,
            }, checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                break

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model.eval().cpu(), history, checkpoint_path

variant_names = {
    "hard": "Hard-label BCE",
    "soft": "Soft-label BCE",
    "brier_ablation": "Soft BCE + Brier (ablation)",
}
if ENABLE_AMBIGUITY_HEAD:
    variant_names["ambiguity_multitask"] = "Soft BCE + ambiguity head"
if RUN_QUALITY_FILTER_ABLATION:
    variant_names["soft_quality_filtered"] = "Soft BCE (quality-filtered train)"

models_by_seed, histories, checkpoint_paths = {}, {}, {}
for run_seed in EXPERIMENT_SEEDS:
    models_by_seed[run_seed] = {}
    for variant, display_name in variant_names.items():
        variant_train_idx = quality_filtered_train_idx if variant == "soft_quality_filtered" else train_idx
        model, history, checkpoint = train_variant(variant, display_name, run_seed, variant_train_idx)
        models_by_seed[run_seed][variant] = model
        histories[f"seed_{run_seed}/{variant}"] = history
        checkpoint_paths[f"seed_{run_seed}/{variant}"] = str(checkpoint)

PRIMARY_SEED = EXPERIMENT_SEEDS[0]
trained_models = models_by_seed[PRIMARY_SEED]

In [ ]:
primary_histories = {key.split("/", 1)[1]: value for key, value in histories.items() if key.startswith(f"seed_{PRIMARY_SEED}/")}
palette = dict(zip(variant_names, sns.color_palette("colorblind", len(variant_names))))
plt.figure(figsize=(11, 5.5))
for variant, history in primary_histories.items():
    epochs_axis = np.arange(1, len(history["train_loss"]) + 1)
    plt.plot(epochs_axis, history["train_loss"], "--", color=palette[variant], alpha=0.65,
             label=f"{variant_names[variant]} train")
    plt.plot(epochs_axis, history["validation_loss"], color=palette[variant],
             label=f"{variant_names[variant]} validation")
plt.xlabel("Epoch")
plt.ylabel("Training objective")
plt.title(f"Training and validation objectives — seed {PRIMARY_SEED}")
plt.legend(fontsize=8, ncol=2)
save_current_figure("C_training_validation_curves.png")
plt.show()

## 9. Evaluation

Fixed 0.5 and validation-selected per-event thresholds are reported separately. The test split is never used for tuning.

In [ ]:
def predict_outputs(model, indices):
    model = model.to(DEVICE).eval()
    loader = DataLoader(torch.from_numpy(embeddings[indices]).float(), batch_size=max(RUN_BATCH_SIZE, 32))
    event_chunks, ambiguity_chunks = [], []
    with torch.inference_mode():
        for batch in loader:
            output = model(batch.to(DEVICE))
            if isinstance(output, tuple):
                event_logits, ambiguity_logits = output
                ambiguity_chunks.append(torch.sigmoid(ambiguity_logits).cpu().numpy())
            else:
                event_logits = output
            event_chunks.append(torch.sigmoid(event_logits).cpu().numpy())
    model.cpu()
    event_probabilities = np.concatenate(event_chunks)
    ambiguity_probabilities = np.concatenate(ambiguity_chunks) if ambiguity_chunks else None
    return event_probabilities, ambiguity_probabilities

def binary_entropy(probabilities):
    p = np.clip(np.asarray(probabilities), 1e-7, 1 - 1e-7)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

def expected_calibration_error(probabilities, targets, n_bins=10):
    p, y = np.asarray(probabilities).ravel(), np.asarray(targets).ravel()
    if p.shape != y.shape or n_bins < 1:
        raise ValueError("Probabilities and targets must align; n_bins must be positive.")
    edges = np.linspace(0, 1, n_bins + 1)
    ids = np.minimum(np.digitize(p, edges[1:-1], right=False), n_bins - 1)
    return float(sum(
        (ids == b).mean() * abs(p[ids == b].mean() - y[ids == b].mean())
        for b in range(n_bins) if (ids == b).any()
    ))

def safe_rank_metric(function, y_true, y_score):
    if np.unique(y_true).size < 2:
        return np.nan
    try:
        return float(function(y_true, y_score))
    except ValueError:
        return np.nan

def select_f1_thresholds(validation_targets, validation_probabilities, default=0.5):
    validation_targets = np.asarray(validation_targets)
    validation_probabilities = np.asarray(validation_probabilities)
    thresholds = []
    for j in range(validation_targets.shape[1]):
        y, p = validation_targets[:, j], validation_probabilities[:, j]
        if np.unique(y).size < 2:
            thresholds.append(float(default))
            continue
        candidates = np.unique(np.concatenate(([default], p)))
        scores = np.asarray([f1_score(y, p >= threshold, zero_division=0) for threshold in candidates])
        best = candidates[np.flatnonzero(np.isclose(scores, scores.max()))]
        thresholds.append(float(sorted(best, key=lambda value: (abs(value - default), -value))[0]))
    return np.asarray(thresholds)

def confusion_statistics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    divide = lambda numerator, denominator: float(numerator / denominator) if denominator else np.nan
    return {
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "specificity": divide(tn, tn + fp), "npv": divide(tn, tn + fn),
        "false_positive_rate": divide(fp, fp + tn), "false_negative_rate": divide(fn, fn + tp),
    }

def evaluate_predictions(run_seed, variant, probabilities, thresholds, policy):
    y_hard, y_soft = hard_targets[test_idx], soft_targets[test_idx]
    y_pred = (probabilities >= thresholds).astype(int)
    precision, recall, per_f1, support = precision_recall_fscore_support(y_hard, y_pred, average=None, zero_division=0)
    auroc = [safe_rank_metric(roc_auc_score, y_hard[:, j], probabilities[:, j]) for j in range(len(EVENTS))]
    ap = [safe_rank_metric(average_precision_score, y_hard[:, j], probabilities[:, j]) for j in range(len(EVENTS))]
    entropy = binary_entropy(probabilities).mean(axis=1)
    sample_error = np.not_equal(y_pred, y_hard).mean(axis=1)
    ambiguity_fraction = ambiguity_targets[test_idx].mean(axis=1)
    summary = {
        "seed": run_seed, "model": variant, "threshold_policy": policy,
        "macro_f1": float(f1_score(y_hard, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_hard, y_pred, average="micro", zero_division=0)),
        "macro_auroc": float(np.nanmean(auroc)) if np.isfinite(auroc).any() else np.nan,
        "macro_average_precision": float(np.nanmean(ap)) if np.isfinite(ap).any() else np.nan,
        "hard_brier": float(np.mean((probabilities - y_hard) ** 2)),
        "soft_brier": float(np.mean((probabilities - y_soft) ** 2)),
        "hard_ece": expected_calibration_error(probabilities, y_hard),
        "soft_ece": expected_calibration_error(probabilities, y_soft),
        "mean_confidence": float(np.maximum(probabilities, 1 - probabilities).mean()),
        "mean_prediction_entropy": float(entropy.mean()),
        "uncertainty_ambiguity_spearman": float(spearmanr(entropy, ambiguity_fraction).statistic),
    }
    rows = []
    for j, event in enumerate(EVENTS):
        row = {
            "seed": run_seed, "model": variant, "threshold_policy": policy, "event": event,
            "threshold": float(thresholds[j]), "precision": float(precision[j]),
            "recall": float(recall[j]), "f1": float(per_f1[j]), "support": int(support[j]),
            "auroc": auroc[j], "average_precision": ap[j],
            "hard_brier": float(np.mean((probabilities[:, j] - y_hard[:, j]) ** 2)),
            "soft_brier": float(np.mean((probabilities[:, j] - y_soft[:, j]) ** 2)),
            "hard_ece": expected_calibration_error(probabilities[:, j], y_hard[:, j]),
            "soft_ece": expected_calibration_error(probabilities[:, j], y_soft[:, j]),
        }
        row.update(confusion_statistics(y_hard[:, j], y_pred[:, j]))
        rows.append(row)
    return summary, rows, entropy, sample_error

predictions_by_seed, ambiguity_predictions_by_seed = {}, {}
thresholds_by_seed, summary_rows, per_event_rows, threshold_rows = {}, [], [], []
for run_seed, seed_models in models_by_seed.items():
    predictions_by_seed[run_seed], ambiguity_predictions_by_seed[run_seed], thresholds_by_seed[run_seed] = {}, {}, {}
    for variant, model in seed_models.items():
        val_probs, val_ambiguity_probs = predict_outputs(model, val_idx)
        test_probs, test_ambiguity_probs = predict_outputs(model, test_idx)
        predictions_by_seed[run_seed][variant] = {"validation": val_probs, "test": test_probs}
        ambiguity_predictions_by_seed[run_seed][variant] = {"validation": val_ambiguity_probs, "test": test_ambiguity_probs}
        fixed = np.full(len(EVENTS), 0.5)
        tuned = select_f1_thresholds(hard_targets[val_idx], val_probs)
        thresholds_by_seed[run_seed][variant] = {"fixed_0.5": fixed, "validation_tuned": tuned}
        for policy, thresholds in thresholds_by_seed[run_seed][variant].items():
            summary, rows, _, _ = evaluate_predictions(run_seed, variant, test_probs, thresholds, policy)
            summary_rows.append(summary)
            per_event_rows.extend(rows)
            threshold_rows.extend({
                "seed": run_seed, "model": variant, "event": event,
                "policy": policy, "threshold": float(thresholds[j]),
            } for j, event in enumerate(EVENTS))

metrics_by_seed_df = pd.DataFrame(summary_rows)
per_class_df = pd.DataFrame(per_event_rows)
threshold_df = pd.DataFrame(threshold_rows)
calibration_by_event_df = per_class_df[[
    "seed", "model", "threshold_policy", "event", "hard_brier", "soft_brier", "hard_ece", "soft_ece"
]].copy()
metric_columns = ["macro_f1", "micro_f1", "macro_auroc", "macro_average_precision", "hard_brier", "soft_brier", "hard_ece", "soft_ece"]
comparison_df = metrics_by_seed_df.groupby(["model", "threshold_policy"])[metric_columns].agg(["mean", "std"])

test_probabilities = {variant: values["test"] for variant, values in predictions_by_seed[PRIMARY_SEED].items()}
val_probabilities = {variant: values["validation"] for variant, values in predictions_by_seed[PRIMARY_SEED].items()}
thresholds_by_variant = thresholds_by_seed[PRIMARY_SEED]
display(metrics_by_seed_df[metrics_by_seed_df.seed == PRIMARY_SEED])
display(threshold_df[threshold_df.seed == PRIMARY_SEED])
display(calibration_by_event_df[calibration_by_event_df.seed == PRIMARY_SEED])

## 10. Calibration and selective prediction

Test-set analyses cover precision–recall, per-event calibration, ambiguity, and risk–coverage baselines.

In [ ]:
primary_per_class = per_class_df[per_class_df.seed == PRIMARY_SEED]
fig, axes = plt.subplots(1, len(variant_names), figsize=(4.5 * len(variant_names), 4.5), sharey=True)
axes = np.atleast_1d(axes)
for ax, variant in zip(axes, variant_names):
    subset = primary_per_class[primary_per_class.model == variant]
    sns.barplot(data=subset, x="event", y="f1", hue="threshold_policy", errorbar=None, ax=ax)
    ax.set(title=variant_names[variant], xlabel="Event", ylabel="F1", ylim=(0, 1))
    ax.tick_params(axis="x", rotation=25)
fig.suptitle("Per-class F1: fixed vs validation-tuned thresholds")
fig.tight_layout()
save_current_figure("D_per_class_f1.png")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
for j, event in enumerate(EVENTS):
    ax = axes.flat[j]
    y = hard_targets[test_idx, j]
    ax.axhline(y.mean(), color="gray", linestyle="--", linewidth=1, label="Prevalence")
    for variant, probabilities in test_probabilities.items():
        precision, recall, _ = precision_recall_curve(y, probabilities[:, j])
        ap = safe_rank_metric(average_precision_score, y, probabilities[:, j])
        ax.plot(recall, precision, color=palette[variant], label=f"{variant_names[variant]} (AP={ap:.3f})")
    ax.set(title=event, xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
axes.flat[-1].axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", fontsize=8)
fig.suptitle("Per-event precision–recall curves")
fig.tight_layout()
save_current_figure("I_precision_recall_curves.png")
plt.show()

In [ ]:
def reliability_points(probabilities, targets, n_bins=10):
    p, y = np.asarray(probabilities).ravel(), np.asarray(targets).ravel()
    edges = np.linspace(0, 1, n_bins + 1)
    ids = np.minimum(np.digitize(p, edges[1:-1]), n_bins - 1)
    rows = [(p[ids == b].mean(), y[ids == b].mean(), (ids == b).sum()) for b in range(n_bins) if (ids == b).any()]
    return tuple(np.asarray(values) for values in zip(*rows)) if rows else (np.array([]),) * 3

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, target, title in zip(
    axes, [hard_targets[test_idx], soft_targets[test_idx]],
    ["Majority-vote targets", "Annotator-proportion targets"],
):
    ax.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect calibration")
    for variant, probabilities in test_probabilities.items():
        x, observed, _ = reliability_points(probabilities, target)
        ax.plot(x, observed, marker="o", color=palette[variant], label=variant_names[variant])
    ax.set(title=title, xlabel="Mean probability", ylabel="Observed target", xlim=(0, 1), ylim=(0, 1))
axes[1].legend(fontsize=7)
fig.suptitle("Pooled reliability diagrams")
fig.tight_layout()
save_current_figure("E_reliability_diagrams.png")
plt.show()

primary_calibration = calibration_by_event_df[
    (calibration_by_event_df.seed == PRIMARY_SEED)
    & (calibration_by_event_df.threshold_policy == "fixed_0.5")
]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric, title in zip(axes, ["hard_ece", "soft_ece"], ["Majority-label ECE", "Soft-target ECE"]):
    table = primary_calibration.pivot(index="event", columns="model", values=metric).reindex(EVENTS)
    sns.heatmap(table, annot=True, fmt=".3f", cmap="viridis_r", ax=ax)
    ax.set_title(title)
fig.tight_layout()
save_current_figure("J_per_event_calibration.png")
plt.show()

In [ ]:
def risk_coverage_curve(sample_errors, uncertainty_scores, coverages):
    errors, scores = np.asarray(sample_errors), np.asarray(uncertainty_scores)
    order = np.argsort(scores, kind="stable")
    return np.asarray([
        errors[order[:max(1, int(math.ceil(float(coverage) * len(order))))]].mean()
        for coverage in coverages
    ])

def random_risk_curve(sample_errors, coverages, repeats=100, seed=0):
    rng, errors = np.random.default_rng(seed), np.asarray(sample_errors)
    curves = []
    for _ in range(repeats):
        curves.append(risk_coverage_curve(errors, rng.random(len(errors)), coverages))
    return np.mean(curves, axis=0)

coverages = np.linspace(0.1, 1.0, 10)
train_prevalence = hard_targets[train_idx].mean(axis=0)
risk_rows = []
for variant, probabilities in test_probabilities.items():
    tuned = thresholds_by_variant[variant]["validation_tuned"]
    predictions = probabilities >= tuned
    sample_errors = np.not_equal(predictions, hard_targets[test_idx]).mean(axis=1)
    scores = {
        "predictive_entropy": binary_entropy(probabilities).mean(axis=1),
        "prevalence_based_rarity": (probabilities * (1 - train_prevalence)[None, :]).mean(axis=1),
    }
    ambiguity_probabilities = ambiguity_predictions_by_seed[PRIMARY_SEED][variant]["test"]
    if ambiguity_probabilities is not None:
        scores["predicted_ambiguity"] = ambiguity_probabilities.mean(axis=1)
    for ranking, score in scores.items():
        risks = risk_coverage_curve(sample_errors, score, coverages)
        risk_rows.extend({"model": variant, "ranking": ranking, "coverage": float(c), "hamming_risk": float(r)} for c, r in zip(coverages, risks))
    random_risks = random_risk_curve(sample_errors, coverages, repeats=100, seed=RANDOM_SEED)
    risk_rows.extend({"model": variant, "ranking": "random", "coverage": float(c), "hamming_risk": float(r)} for c, r in zip(coverages, random_risks))
risk_coverage_df = pd.DataFrame(risk_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for variant in variant_names:
    subset = risk_coverage_df[(risk_coverage_df.model == variant) & (risk_coverage_df.ranking == "predictive_entropy")]
    axes[0].plot(subset.coverage, subset.hamming_risk, marker="o", color=palette[variant], label=variant_names[variant])
baseline_model = "ambiguity_multitask" if "ambiguity_multitask" in variant_names else "soft"
for ranking in risk_coverage_df.ranking.unique():
    subset = risk_coverage_df[(risk_coverage_df.model == baseline_model) & (risk_coverage_df.ranking == ranking)]
    if not subset.empty:
        axes[1].plot(subset.coverage, subset.hamming_risk, marker="o", label=ranking)
axes[0].set(title="Entropy ranking by model", xlabel="Coverage", ylabel="Hamming risk")
axes[1].set(title=f"{variant_names[baseline_model]} ranking baselines", xlabel="Coverage")
axes[0].legend(fontsize=7)
axes[1].legend(fontsize=8)
fig.tight_layout()
save_current_figure("F_risk_coverage.png")
plt.show()

In [ ]:
ambiguity_detection_rows, ambiguity_prevalence_rows = [], []
analysis_variants = list(test_probabilities)
for variant in analysis_variants:
    probabilities = test_probabilities[variant]
    entropy_scores = binary_entropy(probabilities)
    validation_entropy_scores = binary_entropy(val_probabilities[variant])
    entropy_thresholds = select_f1_thresholds(ambiguity_targets[val_idx], validation_entropy_scores)
    head_test = ambiguity_predictions_by_seed[PRIMARY_SEED][variant]["test"]
    head_val = ambiguity_predictions_by_seed[PRIMARY_SEED][variant]["validation"]
    head_thresholds = select_f1_thresholds(ambiguity_targets[val_idx], head_val) if head_test is not None else None
    train_ambiguity_prevalence = ambiguity_targets[train_idx].mean(axis=0)
    prevalence_validation_scores = np.broadcast_to(
        train_ambiguity_prevalence, (len(val_idx), len(EVENTS))
    )
    prevalence_thresholds = select_f1_thresholds(
        ambiguity_targets[val_idx], prevalence_validation_scores
    )
    for j, event in enumerate(EVENTS):
        target = ambiguity_targets[test_idx, j]
        methods = {
            "event_entropy": (entropy_scores[:, j], entropy_thresholds[j]),
            "train_ambiguity_prevalence": (
                np.full(len(target), train_ambiguity_prevalence[j]), prevalence_thresholds[j]
            ),
        }
        if head_test is not None:
            methods["ambiguity_head"] = (head_test[:, j], head_thresholds[j])
        for method, (score, threshold) in methods.items():
            ambiguity_detection_rows.append({
                "model": variant, "event": event, "method": method,
                "ambiguity_prevalence": float(target.mean()),
                "auroc": safe_rank_metric(roc_auc_score, target, score),
                "average_precision": safe_rank_metric(average_precision_score, target, score),
                "brier": float(np.mean((score - target) ** 2)),
                "f1": float(f1_score(target, score >= threshold, zero_division=0)),
                "threshold": float(threshold),
            })
        ambiguity_prevalence_rows.append({
            "model": variant, "event": event,
            "event_prevalence_train": float(hard_targets[train_idx, j].mean()),
            "ambiguity_prevalence_test": float(target.mean()),
            "entropy_ambiguity_spearman": float(spearmanr(entropy_scores[:, j], target).statistic),
        })
ambiguity_detection_df = pd.DataFrame(ambiguity_detection_rows)
ambiguity_prevalence_df = pd.DataFrame(ambiguity_prevalence_rows)
display(ambiguity_detection_df)
display(ambiguity_prevalence_df)

ambiguity_fraction = ambiguity_targets[test_idx].mean(axis=1)
plt.figure(figsize=(9, 5))
for variant, probabilities in test_probabilities.items():
    entropy = binary_entropy(probabilities).mean(axis=1)
    grouped = pd.DataFrame({"ambiguity": ambiguity_fraction, "entropy": entropy}).groupby("ambiguity", as_index=False).agg(mean_entropy=("entropy", "mean"), n=("entropy", "size"))
    plt.plot(grouped.ambiguity, grouped.mean_entropy, marker="o", color=palette[variant], label=variant_names[variant])
plt.xlabel("Fraction of ambiguous event targets")
plt.ylabel("Mean prediction entropy (bits)")
plt.title("Aggregate ambiguity versus entropy")
plt.legend(fontsize=8)
save_current_figure("G_disagreement_vs_uncertainty.png")
plt.show()

In [ ]:
primary_tuned = per_class_df[
    (per_class_df.seed == PRIMARY_SEED) & (per_class_df.threshold_policy == "validation_tuned")
]
fig, axes = plt.subplots(1, len(variant_names), figsize=(4.5 * len(variant_names), 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, variant in zip(axes, variant_names):
    panel = primary_tuned[primary_tuned.model == variant].set_index("event")[["false_positive_rate", "false_negative_rate"]].reindex(EVENTS)
    sns.heatmap(panel, annot=True, fmt=".3f", cmap="magma", vmin=0, ax=ax, cbar=ax is axes[-1])
    ax.set_title(variant_names[variant])
fig.suptitle("Validation-tuned confusion rates")
fig.tight_layout()
save_current_figure("H_multilabel_error_analysis.png")
plt.show()

quality_rows = []
subgroups = {"all": np.ones(len(test_idx), dtype=bool), "clean_all_flags": ~quality_any_flag[test_idx]}
for name in quality_columns_present:
    values = metadata.iloc[test_idx][name].fillna(0).to_numpy()
    subgroups[f"{name}_clear"] = values < QUALITY_FILTER_MIN_COUNT
    subgroups[f"{name}_flagged"] = values >= QUALITY_FILTER_MIN_COUNT
for variant, probabilities in test_probabilities.items():
    thresholds = thresholds_by_variant[variant]["validation_tuned"]
    predictions = probabilities >= thresholds
    for subgroup, mask in subgroups.items():
        if not mask.any():
            continue
        quality_rows.append({
            "model": variant, "subgroup": subgroup, "n_clips": int(mask.sum()),
            "macro_f1": float(f1_score(hard_targets[test_idx][mask], predictions[mask], average="macro", zero_division=0)),
            "hamming_error": float(np.not_equal(predictions[mask], hard_targets[test_idx][mask]).mean()),
            "soft_brier": float(np.mean((probabilities[mask] - soft_targets[test_idx][mask]) ** 2)),
        })
quality_control_df = pd.DataFrame(quality_rows)
display(quality_control_df)

## 11. Uncertainty intervals

Seed summaries and paired bootstrap intervals preserve known dependence groups.

In [ ]:
def paired_group_bootstrap_difference(
    y_hard, y_soft, reference_probabilities, candidate_probabilities,
    groups, reference_thresholds, candidate_thresholds, replicates, seed, confidence=0.95,
):
    y_hard, y_soft, groups = np.asarray(y_hard), np.asarray(y_soft), np.asarray(groups)
    reference_probabilities = np.asarray(reference_probabilities)
    candidate_probabilities = np.asarray(candidate_probabilities)
    if not (len(y_hard) == len(y_soft) == len(groups) == len(reference_probabilities) == len(candidate_probabilities)):
        raise ValueError("Targets, probabilities, and groups must have the same sample count.")
    if replicates < 1 or not 0 < confidence < 1:
        raise ValueError("replicates must be positive and confidence must be between 0 and 1.")
    unique_groups = np.unique(groups)
    members = {group: np.flatnonzero(groups == group) for group in unique_groups}
    rng = np.random.default_rng(seed)
    f1_deltas, brier_deltas = [], []
    for _ in range(replicates):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = np.concatenate([members[group] for group in sampled_groups])
        reference_f1 = f1_score(y_hard[indices], reference_probabilities[indices] >= reference_thresholds, average="macro", zero_division=0)
        candidate_f1 = f1_score(y_hard[indices], candidate_probabilities[indices] >= candidate_thresholds, average="macro", zero_division=0)
        f1_deltas.append(candidate_f1 - reference_f1)
        reference_brier = np.mean((reference_probabilities[indices] - y_soft[indices]) ** 2)
        candidate_brier = np.mean((candidate_probabilities[indices] - y_soft[indices]) ** 2)
        brier_deltas.append(candidate_brier - reference_brier)
    alpha = (1 - confidence) / 2
    reference_f1 = f1_score(
        y_hard, reference_probabilities >= reference_thresholds,
        average="macro", zero_division=0,
    )
    candidate_f1 = f1_score(
        y_hard, candidate_probabilities >= candidate_thresholds,
        average="macro", zero_division=0,
    )
    observed = {
        "macro_f1_delta": float(candidate_f1 - reference_f1),
        "soft_brier_delta": float(
            np.mean((candidate_probabilities - y_soft) ** 2)
            - np.mean((reference_probabilities - y_soft) ** 2)
        ),
    }
    summarize = lambda values, estimate: {
        "estimate": estimate,
        "lower": float(np.quantile(values, alpha)),
        "upper": float(np.quantile(values, 1 - alpha)),
    }
    return {
        "macro_f1_delta": summarize(f1_deltas, observed["macro_f1_delta"]),
        "soft_brier_delta": summarize(brier_deltas, observed["soft_brier_delta"]),
    }

bootstrap_rows = []
reference_variant = "hard"
test_groups = dependence_groups[test_idx]
for candidate in variant_names:
    if candidate == reference_variant:
        continue
    result = paired_group_bootstrap_difference(
        hard_targets[test_idx], soft_targets[test_idx],
        test_probabilities[reference_variant], test_probabilities[candidate], test_groups,
        thresholds_by_variant[reference_variant]["validation_tuned"],
        thresholds_by_variant[candidate]["validation_tuned"],
        RUN_BOOTSTRAP_REPLICATES, RANDOM_SEED, BOOTSTRAP_CONFIDENCE,
    )
    for metric, interval in result.items():
        bootstrap_rows.append({"reference": reference_variant, "candidate": candidate, "metric": metric, **interval})
bootstrap_df = pd.DataFrame(bootstrap_rows)
seed_summary_df = metrics_by_seed_df.groupby(["model", "threshold_policy"])[[
    "macro_f1", "micro_f1", "macro_auroc", "macro_average_precision", "hard_brier", "soft_brier", "hard_ece", "soft_ece"
]].agg(["mean", "std", "count"])
display(seed_summary_df)
display(bootstrap_df)

primary_tuned_summary = metrics_by_seed_df[
    (metrics_by_seed_df.seed == PRIMARY_SEED) & (metrics_by_seed_df.threshold_policy == "validation_tuned")
].set_index("model")
best_model = primary_tuned_summary.macro_f1.idxmax()
summary_text = f"""# StutterSense experiment summary

- Dataset/model revisions: `{RESOLVED_DATASET_REVISION}` / `{RESOLVED_MODEL_REVISION}`.
- Sampling: {sampling_metadata["algorithm"]}; full stream = {sampling_metadata["full_stream"]}.
- Split scope: {split_scope}.
- Optimization seeds: {EXPERIMENT_SEEDS}.
- Highest validation-tuned macro F1 for the primary seed: **{best_model}** ({primary_tuned_summary.loc[best_model, "macro_f1"]:.4f}).
- The BCE+Brier variant is reported only as a regularization ablation.

These measurements are generated by the current run. They do not establish clinical validity, speaker-independent generalization, or a pure measure of epistemic uncertainty.
"""
display(Markdown(summary_text))

## 12. Interactive inference

Uploaded audio is chunked with overlap and bounded by configurable size and duration limits.

In [ ]:
import ipywidgets as widgets

def chunk_waveform(waveform, sample_rate=SAMPLE_RATE, chunk_seconds=UPLOAD_CHUNK_SECONDS, overlap_seconds=UPLOAD_CHUNK_OVERLAP_SECONDS):
    chunk_size = int(chunk_seconds * sample_rate)
    step = int((chunk_seconds - overlap_seconds) * sample_rate)
    if chunk_size <= 0 or step <= 0:
        raise ValueError("Chunk duration must be positive and exceed overlap.")
    return [(start, waveform[start:start + chunk_size]) for start in range(0, len(waveform), step) if len(waveform[start:start + chunk_size])]

def embedding_for_waveform(waveform):
    encoder.to(DEVICE).eval()
    inputs = feature_extractor([waveform], sampling_rate=SAMPLE_RATE, padding=True, truncation=False,
                               return_attention_mask=True, return_tensors="pt")
    values = inputs["input_values"].to(DEVICE)
    mask = inputs.get("attention_mask", torch.ones_like(values, dtype=torch.long)).to(DEVICE)
    with torch.inference_mode():
        hidden = encoder(input_values=values, attention_mask=mask).last_hidden_state
        pooled = masked_mean_pool(hidden, mask, encoder)
    encoder.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pooled.float().cpu().numpy()

def predict_embedding(single_embedding):
    tensor, output = torch.from_numpy(single_embedding).float(), {}
    for variant, model in trained_models.items():
        model.eval()
        with torch.inference_mode():
            result = model(tensor)
            event_logits, ambiguity_logits = result if isinstance(result, tuple) else (result, None)
            output[variant] = {
                "events": torch.sigmoid(event_logits).numpy()[0],
                "ambiguity": None if ambiguity_logits is None else torch.sigmoid(ambiguity_logits).numpy()[0],
            }
    return output

def render_demo(waveform, title, true_counts=None, precomputed_embedding=None):
    clear_output(wait=True)
    duration = len(waveform) / SAMPLE_RATE
    display(Markdown(f"### {title}\nDuration: {duration:.2f} seconds"))
    display(IPythonAudio(waveform, rate=SAMPLE_RATE))
    plot_step = max(1, len(waveform) // 100_000)
    time_axis = np.arange(0, len(waveform), plot_step) / SAMPLE_RATE
    plt.figure(figsize=(10, 2.5))
    plt.plot(time_axis, waveform[::plot_step], linewidth=0.7)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Waveform")
    plt.show()
    if true_counts is not None:
        display(pd.DataFrame({"event": EVENTS, "annotator_count": true_counts, "soft_target": np.asarray(true_counts) / 3}))

    if precomputed_embedding is not None:
        chunks = [(0, duration, predict_embedding(precomputed_embedding))]
    else:
        if duration > UPLOAD_CHUNK_SECONDS:
            display(Markdown(f"**Memory safeguard:** processed as {UPLOAD_CHUNK_SECONDS}-second chunks with {UPLOAD_CHUNK_OVERLAP_SECONDS}-second overlap."))
        chunks = []
        for start, chunk in chunk_waveform(waveform):
            chunks.append((start / SAMPLE_RATE, (start + len(chunk)) / SAMPLE_RATE, predict_embedding(embedding_for_waveform(chunk))))

    rows = []
    for start, end, predictions in chunks:
        for variant, outputs in predictions.items():
            row = {"start_s": start, "end_s": end, "model": variant,
                   "mean_event_entropy": float(binary_entropy(outputs["events"]).mean())}
            row.update(dict(zip(EVENTS, outputs["events"])))
            if outputs["ambiguity"] is not None:
                row["mean_predicted_ambiguity"] = float(outputs["ambiguity"].mean())
            rows.append(row)
    display(pd.DataFrame(rows).style.format(precision=3))
    display(Markdown("*Experimental research predictions only; not a clinical diagnosis.*"))

clip_selector = widgets.Dropdown(
    options=[(f"Clip {i}: {metadata.loc[i, 'duration_seconds']:.2f}s", i) for i in range(len(waveforms))],
    value=0, description="Clip:", layout=widgets.Layout(width="55%"),
)
clip_output = widgets.Output()
def update_clip(change=None):
    i = int(clip_selector.value)
    with clip_output:
        render_demo(waveforms[i], f"Loaded clip {i}", counts_matrix[i].astype(int), embeddings[i:i + 1])
clip_selector.observe(update_clip, names="value")
display(clip_selector, clip_output)
update_clip()

In [ ]:
upload_widget = widgets.FileUpload(accept=".wav,audio/*", multiple=False, description="Upload audio")
upload_output = widgets.Output()

def uploaded_bytes(widget_value):
    item = next(iter(widget_value.values())) if isinstance(widget_value, dict) else widget_value[0]
    content = item["content"]
    return content.tobytes() if hasattr(content, "tobytes") else bytes(content)

def handle_upload(change):
    if not upload_widget.value:
        return
    with upload_output:
        try:
            payload = uploaded_bytes(upload_widget.value)
            if len(payload) > MAX_UPLOAD_MB * 1024 * 1024:
                raise ValueError(f"Upload exceeds the {MAX_UPLOAD_MB} MB safety limit.")
            waveform, original_rate = decode_audio_value({"bytes": payload, "path": None})
            duration = len(waveform) / SAMPLE_RATE
            if duration > MAX_UPLOAD_TOTAL_SECONDS:
                raise ValueError(f"Audio is {duration:.1f}s; limit is {MAX_UPLOAD_TOTAL_SECONDS}s. Split it before upload.")
            render_demo(waveform, f"Uploaded audio (source rate {original_rate} Hz)")
        except Exception as exc:
            clear_output(wait=True)
            print("Could not process uploaded audio:", repr(exc))

upload_widget.observe(handle_upload, names="value")
display(Markdown("### Optional audio upload"), upload_widget, upload_output)

## 13. Optional FluencyBank evaluation

Disabled by default. Execution requires authorized local audio plus verified source, revision, mapping checksum, speaker identity, and access metadata.

In [ ]:
FLUENCYBANK_SOURCE_REPOSITORY = "https://github.com/apple-aiml-research/ml-stuttering-events-dataset"
FLUENCYBANK_REQUIRED_COLUMNS = [
    "audio_path", "sample_id", "speaker_id", *EVENTS,
    "source_repository", "source_revision", "mapping_path", "mapping_sha256",
    "mapping_verified", "license_acknowledged", "access_basis",
]

def validate_fluencybank_manifest(path):
    manifest_path = Path(path)
    if not manifest_path.exists():
        raise FileNotFoundError(manifest_path)
    frame = pd.read_csv(manifest_path)
    missing = [name for name in FLUENCYBANK_REQUIRED_COLUMNS if name not in frame.columns]
    if missing:
        raise ValueError(f"FluencyBank manifest is missing: {missing}")
    if not (frame["source_repository"] == FLUENCYBANK_SOURCE_REPOSITORY).all():
        raise ValueError("Unverified FluencyBank annotation source.")
    revisions = frame["source_revision"].astype(str).str.lower()
    valid_revisions = revisions.map(
        lambda value: len(value) == 40 and all(character in "0123456789abcdef" for character in value)
    )
    if not valid_revisions.all():
        raise ValueError("Every row requires an exact 40-character source commit SHA.")
    for flag in ["mapping_verified", "license_acknowledged"]:
        affirmed = frame[flag].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})
        if not affirmed.all():
            raise PermissionError(f"Every row must affirm {flag}.")
    for identity_column in ["sample_id", "speaker_id", "access_basis"]:
        values = frame[identity_column]
        if values.isna().any() or not (values.astype(str).str.strip().str.len() > 0).all():
            raise ValueError(f"Non-empty {identity_column} values are required.")
    if frame["sample_id"].astype(str).duplicated().any():
        raise ValueError("External sample IDs must be unique.")
    for event in EVENTS:
        values = frame[event].to_numpy()
        if not np.isin(values, [0, 1, 2, 3]).all():
            raise ValueError(f"Invalid {event} counts in external manifest.")
    manifest_root = manifest_path.resolve().parent
    for column in ["audio_path", "mapping_path"]:
        frame[column] = frame[column].map(
            lambda value: str((manifest_root / str(value)).resolve())
            if not Path(str(value)).is_absolute() else str(Path(str(value)).resolve())
        )
        for referenced_path in frame[column]:
            if not Path(referenced_path).is_file():
                raise FileNotFoundError(referenced_path)
    declared_hashes = frame["mapping_sha256"].astype(str).str.lower()
    valid_hashes = declared_hashes.map(
        lambda value: len(value) == 64 and all(character in "0123456789abcdef" for character in value)
    )
    if not valid_hashes.all():
        raise ValueError("Every mapping_sha256 must be a 64-character SHA-256 digest.")
    for mapping_path, expected_hash in set(zip(frame["mapping_path"], declared_hashes)):
        observed_hash = hashlib.sha256(Path(mapping_path).read_bytes()).hexdigest()
        if observed_hash != expected_hash:
            raise ValueError(f"Mapping checksum mismatch: {mapping_path}")
    return frame

external_evaluation_status = {
    "enabled": ENABLE_EXTERNAL_FLUENCYBANK, "executed": False,
    "reason": "Disabled by default; authorized audio and a verified manifest are required.",
}
if ENABLE_EXTERNAL_FLUENCYBANK:
    if not FLUENCYBANK_MANIFEST_PATH:
        raise ValueError("Set FLUENCYBANK_MANIFEST_PATH after verifying access, identity, and mapping.")
    external_manifest = validate_fluencybank_manifest(FLUENCYBANK_MANIFEST_PATH)
    external_rows = []
    for _, row in tqdm(external_manifest.iterrows(), total=len(external_manifest), desc="External FluencyBank inference"):
        waveform, _ = decode_audio_value(str(row.audio_path))
        if len(waveform) / SAMPLE_RATE > UPLOAD_CHUNK_SECONDS:
            raise ValueError("External manifest rows must map to bounded clips; long recordings are not accepted.")
        predictions = predict_embedding(embedding_for_waveform(waveform))
        for variant, outputs in predictions.items():
            result = {"sample_id": row.sample_id, "speaker_id": row.speaker_id, "model": variant}
            result.update({f"prob_{event}": float(outputs["events"][j]) for j, event in enumerate(EVENTS)})
            result.update({f"count_{event}": int(row[event]) for event in EVENTS})
            external_rows.append(result)
    external_predictions_df = pd.DataFrame(external_rows)
    external_predictions_df.to_csv(OUTPUT_PATH / "fluencybank_external_predictions.csv", index=False)
    external_metric_rows = []
    external_counts = external_manifest[EVENTS].to_numpy(dtype=np.float32)
    external_hard = (external_counts >= 2).astype(int)
    external_soft = external_counts / 3.0
    for variant in trained_models:
        model_rows = external_predictions_df[external_predictions_df.model == variant]
        probabilities = model_rows[[f"prob_{event}" for event in EVENTS]].to_numpy()
        for policy, thresholds in thresholds_by_variant[variant].items():
            predictions = probabilities >= thresholds
            external_metric_rows.append({
                "model": variant, "threshold_policy": policy, "n_clips": len(probabilities),
                "macro_f1": float(f1_score(external_hard, predictions, average="macro", zero_division=0)),
                "micro_f1": float(f1_score(external_hard, predictions, average="micro", zero_division=0)),
                "hard_brier": float(np.mean((probabilities - external_hard) ** 2)),
                "soft_brier": float(np.mean((probabilities - external_soft) ** 2)),
            })
    external_metrics_df = pd.DataFrame(external_metric_rows)
    external_metrics_df.to_csv(OUTPUT_PATH / "fluencybank_external_metrics.csv", index=False)
    external_evaluation_status = {
        "enabled": True, "executed": True, "n_manifest_clips": len(external_manifest),
        "source_revisions": sorted(external_manifest.source_revision.astype(str).unique().tolist()),
        "note": "Metrics use thresholds selected only on the internal validation split; speaker IDs are retained for grouped follow-up analyses.",
    }
print(external_evaluation_status)

## 14. Export

Exports include configuration, revisions, identifiers, splits, metrics, figures, histories, and provenance.

In [ ]:
config_export = {
    "dataset_id": DATASET_ID, "dataset_config": DATASET_CONFIG, "dataset_split": "train",
    "dataset_revision_requested": DATASET_REVISION, "dataset_revision_resolved": RESOLVED_DATASET_REVISION,
    "model_name": MODEL_NAME, "model_revision_requested": MODEL_REVISION, "model_revision_resolved": RESOLVED_MODEL_REVISION,
    "max_samples": MAX_SAMPLES, "run_max_samples": RUN_MAX_SAMPLES,
    "full_stream_sampling": RUN_STREAM_SCAN_LIMIT is None, "stream_scan_limit": RUN_STREAM_SCAN_LIMIT,
    "smoke_stream_scan_limit": SMOKE_STREAM_SCAN_LIMIT,
    "sample_rate": SAMPLE_RATE, "random_seed": RANDOM_SEED, "experiment_seeds": EXPERIMENT_SEEDS,
    "batch_size": BATCH_SIZE, "run_batch_size": RUN_BATCH_SIZE,
    "epochs": EPOCHS, "run_epochs": RUN_EPOCHS, "learning_rate": LEARNING_RATE,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE, "dropout": DROPOUT,
    "brier_lambda": BRIER_LAMBDA, "enable_ambiguity_head": ENABLE_AMBIGUITY_HEAD,
    "ambiguity_loss_weight": AMBIGUITY_LOSS_WEIGHT,
    "run_quality_filter_ablation": RUN_QUALITY_FILTER_ABLATION,
    "quality_filter_min_count": QUALITY_FILTER_MIN_COUNT,
    "events": EVENTS, "diagnostic_columns": DIAGNOSTIC_COLUMNS,
    "verified_group_columns": VERIFIED_GROUP_COLUMNS,
    "sample_identifier_columns": SAMPLE_IDENTIFIER_COLUMNS,
    "bootstrap_replicates": RUN_BOOTSTRAP_REPLICATES, "bootstrap_confidence": BOOTSTRAP_CONFIDENCE,
    "verified_group_column": verified_group_column, "split_scope": split_scope,
    "fast_smoke_test": FAST_SMOKE_TEST, "cpu_smoke_test": CPU_SMOKE_TEST,
    "external_fluencybank_enabled": ENABLE_EXTERNAL_FLUENCYBANK,
    "external_fluencybank_manifest": FLUENCYBANK_MANIFEST_PATH,
    "max_upload_mb": MAX_UPLOAD_MB, "max_upload_total_seconds": MAX_UPLOAD_TOTAL_SECONDS,
    "upload_chunk_seconds": UPLOAD_CHUNK_SECONDS,
    "upload_chunk_overlap_seconds": UPLOAD_CHUNK_OVERLAP_SECONDS,
    "show_output_zip_link": SHOW_OUTPUT_ZIP_LINK,
    "output_dir": str(OUTPUT_PATH.resolve()),
}
provenance = {
    "dataset_id": DATASET_ID, "configuration": DATASET_CONFIG, "split": "train",
    "resolved_dataset_revision": RESOLVED_DATASET_REVISION,
    "resolved_model_revision": RESOLVED_MODEL_REVISION,
    "dataset_card_license": DATASET_CARD_LICENSE, "model_card_license": MODEL_CARD_LICENSE,
    "streaming": True, "sampling": sampling_metadata,
    "verified_columns": available_columns, "verified_identifier_columns": verified_identifier_columns,
    "verified_group_column": verified_group_column, "split_scope": split_scope,
    "audio_feature_repr": repr(raw_stream.features.get("audio")),
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(), "python_executable": sys.executable,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "package_versions": {
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "datasets": datasets.__version__,
        "huggingface_hub": __import__("huggingface_hub").__version__,
        "numpy": np.__version__, "scipy": scipy.__version__, "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__, "librosa": librosa.__version__, "soundfile": sf.__version__,
    },
}

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

(OUTPUT_PATH / "config.json").write_text(json.dumps(json_safe(config_export), indent=2))
(OUTPUT_PATH / "dataset_provenance.json").write_text(json.dumps(json_safe(provenance), indent=2))
(OUTPUT_PATH / "split_indices.json").write_text(json.dumps(json_safe(split_indices), indent=2))
(OUTPUT_PATH / "thresholds.json").write_text(json.dumps(json_safe(thresholds_by_seed), indent=2))
(OUTPUT_PATH / "training_histories.json").write_text(json.dumps(json_safe(histories), indent=2))
(OUTPUT_PATH / "external_evaluation_status.json").write_text(json.dumps(json_safe(external_evaluation_status), indent=2))
(OUTPUT_PATH / "evaluation_metrics.json").write_text(json.dumps(json_safe({
    "by_seed": summary_rows, "per_class": per_event_rows,
    "bootstrap": bootstrap_rows, "ambiguity": ambiguity_detection_rows,
}), indent=2))

metrics_by_seed_df.to_csv(OUTPUT_PATH / "metrics_by_seed.csv", index=False)
per_class_df.to_csv(OUTPUT_PATH / "per_class_metrics.csv", index=False)
threshold_df.to_csv(OUTPUT_PATH / "thresholds.csv", index=False)
calibration_by_event_df.to_csv(OUTPUT_PATH / "calibration_by_event.csv", index=False)
risk_coverage_df.to_csv(OUTPUT_PATH / "risk_coverage.csv", index=False)
ambiguity_detection_df.to_csv(OUTPUT_PATH / "ambiguity_detection.csv", index=False)
ambiguity_prevalence_df.to_csv(OUTPUT_PATH / "ambiguity_prevalence.csv", index=False)
quality_control_df.to_csv(OUTPUT_PATH / "quality_control.csv", index=False)
bootstrap_df.to_csv(OUTPUT_PATH / "paired_group_bootstrap.csv", index=False)
metadata.to_csv(OUTPUT_PATH / "sample_manifest.csv", index=False)
(OUTPUT_PATH / "experiment_summary.md").write_text(summary_text)

zip_base = str(OUTPUT_PATH.parent / "stuttersense_outputs")
zip_path = Path(shutil.make_archive(zip_base, "zip", root_dir=OUTPUT_PATH))
print("Output directory:", OUTPUT_PATH)
print("ZIP archive:", zip_path)
if SHOW_OUTPUT_ZIP_LINK:
    display(FileLink(str(zip_path), result_html_prefix="Download outputs: "))